# GPU training on Gaia DR3 with CatalogDataset

This notebook trains a small classifier on Gaia DR3 with `CatalogDataset` and
`CatalogDataLoader`, and measures how much the LSDB side of the pipeline holds back the GPU.

It is **pre-executed**: it needs a CUDA GPU, network access to `data.lsdb.io` (or a local copy of
the catalog), and roughly an hour to run. The hardware it ran on is printed below.

The notebook covers:

1. **Data**: about 1.8 billion Gaia DR3 sources. We keep a handful of photometric and astrometric
   columns and label each source as a star, galaxy or quasar.
2. **Model**: an MLP that predicts the class from magnitudes, colors, parallax and proper motion.
   It does no real science. It is a stand-in workload with a tunable GPU cost.
3. **Benchmark**: the two ways of feeding the GPU (mode A and mode B, below), each at several
   CPU worker counts.
4. **Training**: a few epochs with the faster mode, then an evaluation on held-out sky.

## Two ways to feed the GPU

`CatalogDataset` streams partitions with `lsdb.streams.CatalogStream`, and either dask or PyTorch can
supply the parallelism.

**Mode A: a dask client and one torch process.**
`CatalogDataset(..., client=client)` with `CatalogDataLoader(dataset, num_workers=0)`.

- The dask workers read and filter the `partitions_per_chunk` partitions of a chunk in parallel.
- `CatalogStream` keeps **one chunk** in flight. While the training loop works through chunk *i*,
  chunk *i + 1* is computed. Parallelism across dask workers therefore depends on
  `partitions_per_chunk`, and we set it to at least the number of workers.
- The finished chunk is concatenated on a dask worker and sent to the main process.
- The main process shuffles the chunk, cuts it into batches, converts them to tensors, and runs
  the training step. This per-batch Python work shares one interpreter (and its GIL) with the
  training loop.

**Mode B: the synchronous dask scheduler in several torch workers.**
`CatalogDataLoader(dataset, num_workers=k)` with no client.

- Each torch worker process owns a disjoint shard of partitions. It computes them one at a time
  with dask's synchronous scheduler, then shuffles, batches and converts them to tensors.
- PyTorch keeps `prefetch_factor` finished batches per worker waiting in a queue.
- Workers are started with `spawn`, because fsspec's HTTP filesystem cannot be used in a forked
  process. Each spawned worker imports torch and LSDB, which adds startup time.
- The main process only receives tensors, copies them to the GPU, and trains.

For a given worker count, both modes use the same number of CPU processes (`k` dask workers or `k`
torch workers, plus the main process).

### Known issues with mode B

Two problems affect `CatalogDataLoader(num_workers > 0)`. This notebook works around both, but they
should be fixed in the library.

**1. Forked workers cannot read remote catalogs.** A catalog opened over HTTP(S) holds an fsspec
filesystem whose event loop was created in the main process. When torch forks a worker, the worker
inherits that filesystem, and fsspec refuses to use it:

```
RuntimeError: This class is not fork-safe
```

`open_catalog` already reads from the server, so this happens for any remote catalog. Fork is the
default start method on Linux before Python 3.14, so mode B fails with default settings. The
existing tests use in-memory catalogs, so they don't catch it.
*Workaround here:* `multiprocessing_context="spawn"`, which gives each worker a fresh filesystem. The
cost is a slower start, because each worker imports torch and LSDB.

**2. Catalogs filtered with `query` cannot be sent to spawned workers.** A spawned worker gets a
pickled copy of the dataset, and `Catalog.query` stores a local lambda that cannot be pickled:

```
AttributeError: Can't get local object 'HealpixDataset.query.<locals>.<lambda>'
```

Together with issue 1, that leaves no start method for a remote catalog filtered with `query`.
*Workaround here:* filter with `map_partitions(npd.NestedFrame.dropna)` instead. A library function
pickles by reference.

Possible fixes: make `spawn` the default in `CatalogDataLoader`, or reset fsspec's filesystem in each
worker; and make `query` (and similar operations) picklable in LSDB.

In [ ]:
import os
import threading
import time
from contextlib import contextmanager, nullcontext

import lsdb
import matplotlib.pyplot as plt
import nested_pandas as npd
import numpy as np
import pandas as pd
import torch
from dask.distributed import Client, LocalCluster
from torch import nn

import lsdb_ml
from lsdb_ml import CatalogDataLoader, CatalogDataset

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"lsdb {lsdb.__version__}, lsdb-ml {lsdb_ml.__version__}, torch {torch.__version__}")
N_CPUS = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
print(f"CPU cores available: {N_CPUS}")
print(f"Device: {device}", f"({torch.cuda.get_device_name()})" if device.type == "cuda" else "")

## Configuration

All the settings are in this cell.

- **Data:** `CATALOG_URL` is read over HTTP by default, so network bandwidth is part of what the
  benchmark measures. To measure only LSDB's CPU cost, point it at a local copy on fast disk.
- **Model size:** `HIDDEN` and `DEPTH` set how much work the GPU does per row. A larger model makes
  the GPU slower, so it is easier for the data pipeline to keep up.
- **Benchmark cost:** each benchmark configuration runs for `BENCH_SECONDS` once its first batch
  arrives. The two sweeps (data only, then training) together take about
  `2 × len(CONFIGS) × (BENCH_SECONDS + startup)`.

In [ ]:
CATALOG_URL = "https://data.lsdb.io/hats/gaia_dr3"
SEED = 42

# Data split, in partitions. Gaia DR3 has 2016 partitions with about 0.6M usable rows each.
N_TRAIN_PARTITIONS = 256
N_VAL_PARTITIONS = 32
N_STATS_PARTITIONS = 16  # used to compute feature normalization and class weights

# Model and optimization
BATCH_SIZE = 8192
HIDDEN = 1024
DEPTH = 6
LEARNING_RATE = 1e-3
USE_AMP = device.type == "cuda"  # bfloat16 autocast
EPOCHS = 2

# Data loading
PIN_MEMORY = device.type == "cuda"
PREFETCH_FACTOR = 4  # mode B: batches queued per torch worker

# Benchmark
BENCH_SECONDS = 90  # steady-state measurement window per configuration
WORKER_COUNTS = [n for n in (1, 2, 4, 8, 16, 32) if n < N_CPUS]

# Each configuration is one data pipeline. In mode A, partitions_per_chunk is how many partitions
# the dask workers compute in parallel. Add entries to try other settings, e.g.
# dict(mode="A", workers=8, partitions_per_chunk=16) to give the dask workers more work per chunk.
CONFIGS = []
for n in WORKER_COUNTS:
    CONFIGS.append(dict(mode="A", workers=n, partitions_per_chunk=n))
    CONFIGS.append(dict(mode="B", workers=n, partitions_per_chunk=1))
pd.DataFrame(CONFIGS)

## Data: Gaia DR3

**Features.** G, BP and RP magnitudes and colors, parallax, proper motion, astrometric quality
(RUWE, excess noise), the BP/RP flux excess, and Galactic latitude.

**Labels.** Gaia's Discrete Source Classifier (DSC) gives each source a probability of being a
star, galaxy or quasar (`classprob_dsc_combmod_*`). The label is the most probable class. DSC uses
the BP/RP spectra, which our model does not see, so this is a distillation task: can the summary
photometry and astrometry reproduce the spectra-based classification? Non-stellar sources make up
well under 1% of the data, so the classes are very imbalanced.

`CatalogDataset`'s default collate does not support missing values. We keep only rows where every
column is present. LSDB applies the filter lazily inside each partition's computation, so it is
part of the data-side cost we measure.

In [ ]:
FEATURES = [
    "phot_g_mean_mag",
    "phot_bp_mean_mag",
    "phot_rp_mean_mag",
    "bp_rp",
    "bp_g",
    "g_rp",
    "parallax",
    "parallax_over_error",
    "pmra",
    "pmdec",
    "ruwe",
    "astrometric_excess_noise",
    "phot_bp_rp_excess_factor",
    "b",
]
CLASSES = ["star", "galaxy", "quasar"]
PROB_COLUMNS = [f"classprob_dsc_combmod_{name}" for name in CLASSES]
COLUMNS = FEATURES + PROB_COLUMNS

gaia = lsdb.open_catalog(CATALOG_URL, columns=COLUMNS)
# open_catalog also loads ra/dec, so select only the columns the model uses, then drop rows with
# any missing value. A library function (rather than a lambda or `query`) keeps the catalog
# picklable, which mode B's spawned workers need.
gaia = gaia[COLUMNS].map_partitions(npd.NestedFrame.dropna)
gaia

We split by **partition**, so the validation set is a separate region of the sky from the training
set. That is a stricter test than a random row split, because neighboring sources are correlated.

In [ ]:
order = np.random.default_rng(SEED).permutation(gaia.npartitions)
train_idx = order[:N_TRAIN_PARTITIONS]
val_idx = order[N_TRAIN_PARTITIONS : N_TRAIN_PARTITIONS + N_VAL_PARTITIONS]
stats_idx = train_idx[:N_STATS_PARTITIONS]

train_catalog = gaia.partitions[train_idx.tolist()]
val_catalog = gaia.partitions[val_idx.tolist()]
print(f"train: {train_catalog.npartitions} partitions, val: {val_catalog.npartitions} partitions")

A sample of training partitions gives the feature normalization and the class frequencies.

In [ ]:
%%time
stats_frame = gaia.partitions[stats_idx.tolist()].compute()
class_counts = np.bincount(stats_frame[PROB_COLUMNS].to_numpy().argmax(axis=1), minlength=len(CLASSES))
print(f"{len(stats_frame):,} rows in {N_STATS_PARTITIONS} partitions")
pd.Series(class_counts, index=CLASSES, name="sources")

## Model

Batches from `CatalogDataset` are dictionaries of 1-D tensors, one per column (see
`collate_catalog`). We move the dictionary to the GPU as is and do everything else there:

- `Featurize` turns the raw columns into well-scaled inputs. For example, it takes `asinh` of
  heavy-tailed quantities like parallax and `sin|b|` of the latitude, then standardizes with
  statistics from the sample above.
- `labels` takes the argmax of the DSC probabilities.
- `GaiaClassifier` is a residual MLP. It is deliberately large for 14 inputs (about 25M parameters
  at the defaults), so that the GPU has a non-trivial amount of work per row.

The loss weights each class by the inverse square root of its frequency. That boosts the rare
galaxies and quasars without letting them dominate.

In [ ]:
class Featurize(nn.Module):
    """Turns a batch of raw Gaia columns into standardized model inputs."""

    n_features = 13

    def __init__(self):
        super().__init__()
        self.register_buffer("mean", torch.zeros(self.n_features))
        self.register_buffer("std", torch.ones(self.n_features))

    def transform(self, batch):
        """Raw columns to unnormalized features of shape [B, n_features]."""
        c = {name: batch[name].float() for name in FEATURES}
        pm = torch.hypot(c["pmra"], c["pmdec"])
        return torch.stack(
            [
                c["phot_g_mean_mag"],
                c["phot_bp_mean_mag"],
                c["phot_rp_mean_mag"],
                c["bp_rp"],
                c["bp_g"],
                c["g_rp"],
                torch.asinh(c["parallax"]),
                torch.asinh(c["parallax_over_error"]),
                torch.log1p(pm),
                torch.log(c["ruwe"]),
                torch.log1p(c["astrometric_excess_noise"]),
                torch.log(c["phot_bp_rp_excess_factor"]),
                torch.sin(torch.deg2rad(c["b"])).abs(),
            ],
            dim=1,
        )

    def fit(self, batch):
        """Set the normalization from a sample batch."""
        features = self.transform(batch)
        self.mean.copy_(features.mean(dim=0))
        self.std.copy_(features.std(dim=0))
        return self

    def forward(self, batch):
        """Standardized features of shape [B, n_features]."""
        return (self.transform(batch) - self.mean) / self.std


class Block(nn.Module):
    """Pre-norm residual MLP block."""

    def __init__(self, width):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(width), nn.Linear(width, 2 * width), nn.GELU(), nn.Linear(2 * width, width)
        )

    def forward(self, x):
        """Apply the block with a skip connection."""
        return x + self.net(x)


class GaiaClassifier(nn.Module):
    """Residual MLP from Gaia columns to class logits."""

    def __init__(self, featurize, hidden, depth, n_classes):
        super().__init__()
        self.featurize = featurize
        self.embed = nn.Linear(featurize.n_features, hidden)
        self.blocks = nn.Sequential(*[Block(hidden) for _ in range(depth)])
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, n_classes))

    def forward(self, batch):
        """Class logits of shape [B, n_classes]."""
        return self.head(self.blocks(self.embed(self.featurize(batch))))


def labels(batch):
    """Index of the most probable DSC class."""
    return torch.stack([batch[col] for col in PROB_COLUMNS], dim=1).argmax(dim=1)


def to_device(batch):
    """Copy a batch dictionary to the training device."""
    if device.type == "mps":  # MPS has no float64
        batch = {name: t.float() if t.dtype == torch.float64 else t for name, t in batch.items()}
    return {name: tensor.to(device, non_blocking=True) for name, tensor in batch.items()}


def sync():
    """Wait for queued GPU work, so wall-clock timings include it."""
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


# Normalization from up to 1M sampled rows, converted with the same collate as the loader uses.
sample_batch = lsdb_ml.collate_catalog(
    stats_frame.sample(min(len(stats_frame), 1_000_000), random_state=SEED)
)

frequency = (class_counts + 1) / (class_counts + 1).sum()
class_weights = torch.tensor(frequency**-0.5, dtype=torch.float32)
class_weights = (class_weights / class_weights.mean()).to(device)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)


def new_model():
    """A freshly initialized model and its optimizer."""
    torch.manual_seed(SEED)
    model = GaiaClassifier(Featurize().fit(sample_batch), HIDDEN, DEPTH, len(CLASSES)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    return model, optimizer


def train_step(model, optimizer, batch):
    """One optimization step on a batch; returns the loss."""
    batch = to_device(batch)
    with torch.autocast(device.type, dtype=torch.bfloat16, enabled=USE_AMP):
        loss = loss_fn(model(batch), labels(batch))
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    return loss


if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
model, optimizer = new_model()
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
print(f"class weights: {dict(zip(CLASSES, class_weights.tolist(), strict=True))}")

## Measuring the bottleneck

Three numbers tell us where time goes:

1. **GPU ceiling.** Training steps on one batch that is already on the GPU. No data loading at
   all, so this is the most rows per second the GPU can take for this model and batch size.
2. **Data ceiling.** Iterating the loader without training. This is the most rows per second the
   LSDB pipeline can deliver.
3. **Training.** The real loop. For every step we record how long it **waited** on the loader and
   how long the **step** took, which includes the host-to-device copy and is synchronized so that
   GPU time is counted. The **GPU-busy fraction** is step time divided by wall time. Well below 1
   means the GPU sat idle waiting on data.

If the data side were perfectly overlapped with the GPU, training throughput would be
`min(data ceiling, GPU ceiling)`. When training is slower than both, the two sides compete for
resources. In mode A that is the main process's GIL.

Synchronizing after every step costs a little overlap. For steps of a few milliseconds or more the
effect is small, and in return we can attribute time cleanly. If `nvidia-ml-py` is installed, we
also sample the driver's own GPU utilization as an independent check.

In [ ]:
class GpuUtilization:
    """Samples torch.cuda.utilization() (needs nvidia-ml-py) in a background thread."""

    def __init__(self, interval=0.25):
        self.interval, self.samples = interval, []
        self._stop = threading.Event()
        try:
            torch.cuda.utilization()
            self.available = True
        except Exception:
            self.available = False

    def _run(self):
        while not self._stop.wait(self.interval):
            self.samples.append(torch.cuda.utilization())

    def __enter__(self):
        if self.available:
            self._thread = threading.Thread(target=self._run, daemon=True)
            self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        if self.available:
            self._thread.join()

    @property
    def mean(self):
        """Mean utilization in percent, or NaN without samples."""
        return float(np.mean(self.samples)) if self.samples else np.nan


def gpu_ceiling(model, optimizer, steps=50, warmup=10):
    """Rows per second for training on one batch that is already on the device."""
    batch = to_device(lsdb_ml.collate_catalog(stats_frame.iloc[:BATCH_SIZE]))
    for _ in range(warmup):
        train_step(model, optimizer, batch)
    sync()
    start = time.perf_counter()
    for _ in range(steps):
        train_step(model, optimizer, batch)
    sync()
    elapsed = time.perf_counter() - start
    return dict(rows_per_s=steps * BATCH_SIZE / elapsed, ms_per_step=1e3 * elapsed / steps)


def run(loader, model=None, optimizer=None, seconds=BENCH_SECONDS):
    """Iterate a loader for `seconds` after its first batch, training if a model is given.

    The first batch is excluded from the steady-state numbers and reported as startup time.
    """
    start = time.perf_counter()
    iterator = iter(loader)  # forks torch workers in mode B, so start no threads before this
    batch = next(iterator)
    startup = time.perf_counter() - start

    wait = step = 0.0
    rows = steps = 0
    with GpuUtilization() if model is not None and device.type == "cuda" else nullcontext() as util:
        window_start = time.perf_counter()
        while True:
            t0 = time.perf_counter()
            if model is not None:
                train_step(model, optimizer, batch)
                sync()
            t1 = time.perf_counter()
            step += t1 - t0
            rows += len(batch[COLUMNS[0]])
            steps += 1
            if t1 - window_start >= seconds:
                break
            try:
                batch = next(iterator)
            except StopIteration:
                print("  (data exhausted before the time window ended)")
                break
            wait += time.perf_counter() - t1
        elapsed = time.perf_counter() - window_start
    del iterator  # shuts down torch workers
    return dict(
        startup_s=startup,
        rows_per_s=rows / elapsed,
        gpu_busy=step / elapsed if model is not None else np.nan,
        wait_ms_per_step=1e3 * wait / steps,
        step_ms_per_step=1e3 * step / steps,
        gpu_util_pct=util.mean if model is not None and device.type == "cuda" else np.nan,
        steps=steps,
    )


@contextmanager
def pipeline(mode, workers, partitions_per_chunk, catalog=train_catalog, shuffle=True):
    """A CatalogDataLoader in mode A (dask client) or mode B (torch workers)."""
    dataset_kwargs = dict(
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED,
        partitions_per_chunk=partitions_per_chunk,
        drop_last=shuffle,
    )
    if mode == "A":
        cluster = LocalCluster(
            n_workers=workers, threads_per_worker=1, processes=True, dashboard_address=":0"
        )
        client = Client(cluster)
        try:
            dataset = CatalogDataset(catalog, client=client, **dataset_kwargs)
            yield CatalogDataLoader(dataset, pin_memory=PIN_MEMORY)
        finally:
            client.close()
            cluster.close()
    else:
        dataset = CatalogDataset(catalog, **dataset_kwargs)
        # Spawn, not fork: a forked worker inherits the parent's HTTP filesystem, which fsspec
        # refuses to use after a fork ("This class is not fork-safe").
        yield CatalogDataLoader(
            dataset,
            num_workers=workers,
            prefetch_factor=PREFETCH_FACTOR,
            pin_memory=PIN_MEMORY,
            multiprocessing_context="spawn",
        )

### GPU ceiling

In [ ]:
ceiling = gpu_ceiling(*new_model())
print(f"GPU ceiling: {ceiling['rows_per_s'] / 1e6:.2f}M rows/s ({ceiling['ms_per_step']:.1f} ms per step)")

### Data ceiling

Each configuration iterates the training loader for `BENCH_SECONDS` without touching the GPU.

In [ ]:
data_results = []
for config in CONFIGS:
    print(f"data only: {config}")
    with pipeline(**config) as loader:
        data_results.append({**config, **run(loader)})
data_df = pd.DataFrame(data_results)
data_df[["mode", "workers", "partitions_per_chunk", "startup_s", "rows_per_s", "wait_ms_per_step"]]

### Training throughput

Now the same pipelines feed real training steps. Every configuration uses a fresh model, so the
measurements do not depend on the order they run in. Remote reads can still get faster as the
server's caches warm up. Since mode A and mode B alternate at every worker count, any such drift
affects both modes about equally.

In [ ]:
train_results = []
for config in CONFIGS:
    print(f"training: {config}")
    model, optimizer = new_model()
    with pipeline(**config) as loader:
        train_results.append({**config, **run(loader, model, optimizer)})
train_df = pd.DataFrame(train_results)
train_df["data_ceiling_rows_per_s"] = data_df["rows_per_s"]
train_df["of_gpu_ceiling"] = train_df["rows_per_s"] / ceiling["rows_per_s"]
train_df[
    [
        "mode",
        "workers",
        "partitions_per_chunk",
        "startup_s",
        "rows_per_s",
        "data_ceiling_rows_per_s",
        "of_gpu_ceiling",
        "gpu_busy",
        "gpu_util_pct",
        "wait_ms_per_step",
        "step_ms_per_step",
    ]
]

In [ ]:
MODE_STYLE = {
    "A": dict(color="#2a78d6", label="Mode A: dask client, 1 torch process"),
    "B": dict(color="#eb6834", label="Mode B: synchronous dask, k torch workers"),
}
INK, MUTED = "#0b0b0b", "#52514e"

fig, (ax_rate, ax_busy) = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
for mode, style in MODE_STYLE.items():
    train = train_df[train_df["mode"] == mode]
    data = data_df[data_df["mode"] == mode]
    ax_rate.plot(train["workers"], train["rows_per_s"] / 1e6, "o-", lw=2, ms=8, **style)
    ax_rate.plot(
        data["workers"],
        data["rows_per_s"] / 1e6,
        "o--",
        lw=1.5,
        ms=6,
        mfc="none",
        color=style["color"],
        label=f"{style['label']} (data only)",
    )
    ax_busy.plot(train["workers"], train["gpu_busy"], "o-", lw=2, ms=8, **style)

ax_rate.axhline(ceiling["rows_per_s"] / 1e6, color=MUTED, lw=1.5, ls=":")
ax_rate.annotate(
    "GPU ceiling",
    (WORKER_COUNTS[-1], ceiling["rows_per_s"] / 1e6),
    xytext=(0, 4),
    textcoords="offset points",
    ha="right",
    color=MUTED,
)
ax_rate.set(ylabel="Million rows / s", title="Throughput: training (solid) vs data only (dashed)")
ax_rate.set_ylim(bottom=0)
fig.legend(*ax_rate.get_legend_handles_labels(), loc="outside lower center", ncols=2, frameon=False)

ax_busy.axhline(1, color=MUTED, lw=1, ls=":")
ax_busy.set(ylabel="Fraction of wall time in GPU steps", title="GPU-busy fraction", ylim=(0, 1.05))

for ax in (ax_rate, ax_busy):
    ax.set_xscale("log", base=2)
    ax.set_xticks(WORKER_COUNTS, [str(n) for n in WORKER_COUNTS])
    ax.set_xlabel("CPU workers (dask workers in A, torch workers in B)")
    ax.grid(alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

### Reading the results

- **Where do the dashed lines (data ceiling) cross the GPU ceiling?** That is the number of CPU
  workers this model needs before it stops waiting on data. If they never cross, the pipeline is
  data-bound at every worker count we tried. In that case the useful levers are more CPUs, local
  storage, a bigger model, or larger batches.
- **The gap between a dashed line and its solid line** is what the training loop costs the data
  side. Mode A's batching and tensor conversion run on the main thread, so expect a larger gap
  there. Mode B moves that work into the workers.
- **`wait_ms_per_step` vs `step_ms_per_step`** splits each step into time spent waiting on the
  loader and time spent in the training step itself.
- **`startup_s`** is the time to the first batch. In mode A it includes starting the cluster and
  computing a whole chunk of `partitions_per_chunk` partitions. In mode B it includes forking the
  workers and computing one partition per worker.

The shuffling also differs between the modes. In mode A each chunk mixes `partitions_per_chunk`
regions of sky. In mode B each batch comes from a single partition per worker, so batches from one
worker cover only one patch of sky. Very homogeneous batches can hurt training, but this
benchmark does not measure that.

## Train the model

We train for `EPOCHS` epochs with the configuration that had the highest training throughput above,
and evaluate on the held-out validation partitions after every epoch.

In [ ]:
best = train_df.loc[train_df["rows_per_s"].idxmax(), ["mode", "workers", "partitions_per_chunk"]].to_dict()
best["workers"], best["partitions_per_chunk"] = int(best["workers"]), int(best["partitions_per_chunk"])
print(f"training with {best}")


@torch.no_grad()
def evaluate(model):
    """Confusion matrix (rows: true class, columns: predicted) on the validation partitions."""
    model.eval()
    confusion = torch.zeros(len(CLASSES), len(CLASSES), dtype=torch.int64, device=device)
    with pipeline(**best, catalog=val_catalog, shuffle=False) as loader:
        for batch in loader:
            batch = to_device(batch)
            with torch.autocast(device.type, dtype=torch.bfloat16, enabled=USE_AMP):
                predicted = model(batch).argmax(dim=1)
            confusion += torch.bincount(
                labels(batch) * len(CLASSES) + predicted, minlength=len(CLASSES) ** 2
            ).view(len(CLASSES), len(CLASSES))
    model.train()
    return confusion.cpu().numpy()


def recall(confusion):
    """Per-class recall from a confusion matrix."""
    return dict(zip(CLASSES, np.diag(confusion) / confusion.sum(axis=1).clip(min=1), strict=True))


model, optimizer = new_model()
history = []
for epoch in range(EPOCHS):
    start = time.perf_counter()
    losses, rows = [], 0
    # A pipeline per epoch, so a mode A cluster is not left running during evaluation. A loader that
    # is reused advances the epoch by itself; with a new loader each time, we set it explicitly.
    with pipeline(**best) as loader:
        loader.dataset.set_epoch(epoch)
        for batch in loader:
            losses.append(train_step(model, optimizer, batch).detach())
            rows += len(batch[COLUMNS[0]])
    losses = torch.stack(losses).float().cpu().numpy()
    elapsed = time.perf_counter() - start
    confusion = evaluate(model)
    history.append(dict(epoch=epoch, losses=losses, confusion=confusion))
    recalls = ", ".join(f"{k} {v:.3f}" for k, v in recall(confusion).items())
    print(
        f"epoch {epoch}: {rows:,} rows in {elapsed / 60:.1f} min ({rows / elapsed / 1e6:.2f}M rows/s), "
        f"mean loss {losses.mean():.4f}, val recall: {recalls}"
    )

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

losses = np.concatenate([h["losses"] for h in history])
confusion = history[-1]["confusion"]
# Row-normalized: each true class sums to 1, so the rare classes are readable.
fraction = confusion / confusion.sum(axis=1, keepdims=True).clip(min=1)

fig, (ax_loss, ax_cm) = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
smooth = pd.Series(losses).rolling(200, min_periods=1).mean()
ax_loss.plot(losses, color="#2a78d6", alpha=0.2, lw=1)
ax_loss.plot(smooth, color="#2a78d6", lw=2)
steps_per_epoch = len(history[0]["losses"])
for epoch in range(1, len(history)):
    ax_loss.axvline(epoch * steps_per_epoch, color=MUTED, lw=1, ls=":")
ax_loss.set(
    xlabel="Step", ylabel="Weighted cross-entropy", title="Training loss (200-step mean)", yscale="log"
)
ax_loss.grid(alpha=0.25)
ax_loss.spines[["top", "right"]].set_visible(False)

blues = LinearSegmentedColormap.from_list("blues", ["#f4f8fd", "#2a78d6", "#0d3a73"])
ax_cm.imshow(fraction, cmap=blues, vmin=0, vmax=1)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax_cm.text(
            j,
            i,
            f"{fraction[i, j]:.3f}\n({confusion[i, j]:,})",
            ha="center",
            va="center",
            color="white" if fraction[i, j] > 0.5 else INK,
            fontsize=9,
        )
ax_cm.set(
    xticks=range(len(CLASSES)),
    yticks=range(len(CLASSES)),
    xticklabels=CLASSES,
    yticklabels=CLASSES,
    xlabel="Predicted",
    ylabel="DSC label",
    title="Validation confusion matrix (row-normalized)",
)
plt.show()

## Notes and next steps

- **Isolating LSDB's CPU cost.** Reading over HTTP mixes network bandwidth into the data ceiling.
  Rerun with `CATALOG_URL` pointing at a local copy (for example, the training partitions written
  with `train_catalog.write_catalog(...)`) to see the parse, filter and collate cost alone.
- **Moving the GPU ceiling.** `HIDDEN`, `DEPTH` and `BATCH_SIZE` set the GPU cost per row. A
  benchmark is only informative near the crossover, where neither side completely dominates.
- **Mode A tuning.** `CatalogStream` keeps one chunk in flight, so dask workers sit idle while the
  slowest partition of a chunk finishes. A larger `partitions_per_chunk` (e.g. twice the worker
  count) balances the load better, but needs more memory in the main process.
- **Mode B tuning.** `prefetch_factor` sets how many batches each worker queues. It smooths out
  per-partition stalls, but cannot raise the steady-state throughput.